In [3]:
import os
from pathlib import Path
from ipumspy import IpumsApiClient, MicrodataExtract, readers, ddi

IPUMS_API_KEY = os.environ.get("IPUMS_API_KEY")
ipums = IpumsApiClient(IPUMS_API_KEY)

extract = MicrodataExtract(
    collection = "nhis",
    description = "Sample NHIS extract",
    samples = ["ih2024"],
    variables = ["AGE", "SEX", "URBRRL", "FAMSIZECAT", "FAMKIDNO", "GOTSTAMPFAM", "FSSTATDET"],
)

In [4]:
# Submit the extract request
ipums.submit_extract(extract)
print(f"Extract submitted with id {extract.extract_id}")
#> Extract submitted with id 1

# Wait for the extract to finish
ipums.wait_for_extract(extract)

# Download the extract
DOWNLOAD_DIR = Path('/Users/liza/Documents/brown')
ipums.download_extract(extract, download_dir = DOWNLOAD_DIR)

Extract submitted with id 4


In [8]:
# The XML (DDI) and data files are already downloaded in this folder
DOWNLOAD_DIR = Path.cwd()

# Get the DDI
ddi_file = list(DOWNLOAD_DIR.glob("*.xml"))[0]
ddi = readers.read_ipums_ddi(ddi_file)

# Get the data and turn it into a dataframe
ipums_df = readers.read_microdata(ddi, DOWNLOAD_DIR / ddi.file_description.filename)
ipums_df.head()

/Users/prasannanagarajan/anaconda3/envs/data1030/lib/python3.14/site-packages/ipumspy/readers.py:71: CitationWarning: Use of data from IPUMS is subject to conditions including that users should cite the data appropriately.
See the `ipums_conditions` attribute of this codebook for terms of use.
See the `ipums_citation` attribute of this codebook for the appropriate citation.
  warnings.warn(


,YEAR,SERIAL,STRATA,PSU,NHISHID,URBRRL,PERNUM,NHISPID,HHX,SAMPWEIGHT,ASTATFLG,CSTATFLG,AGE,SEX,FAMSIZECAT,FAMKIDNO
0,2025,1,238,200218,0002025H000004,4,1,0002025H00000410,H000004,7573.110,1,0,81,2,20,0
1,2025,2,236,125092,0002025H000007,2,1,0002025H00000710,H000007,7759.369,1,0,30,2,20,0
2,2025,3,214,125276,0002025H000008,2,1,0002025H00000810,H000008,4409.171,1,0,79,1,1,0
3,2025,4,235,200040,0002025H000010,3,1,0002025H00001010,H000010,12184.900,1,0,26,1,20,0
4,2025,5,222,125154,0002025H000013,3,1,0002025H00001310,H000013,13406.663,1,0,54,2,20,0


In [ ]:
# Save the dataframe so other notebooks (e.g. the EDA notebook) can load it
ipums_df.to_parquet("nhis_data.parquet")
print("Saved nhis_data.parquet")